### 3. Tổn thất Doanh thu do Hệ thống Vận hành (Stockouts)
Những ngày ta thấy có lượng truy cập web traffic cao, khuyến mãi diễn ra, nhưng doanh thu không đạt như kỳ vọng, liệu có phải do đứt gãy tồn kho?

In [ ]:
# Gom nhóm Tồn kho theo từng tháng: tính Tổng số ngày hết hàng của toàn hệ thống (Stockout Days)
monthly_inventory = inventory.groupby(['year', 'month']).agg({'stockout_days':'sum'}).reset_index()
monthly_inventory['Date'] = pd.to_datetime(monthly_inventory[['year', 'month']].assign(DAY=1))
monthly_inventory = monthly_inventory.set_index('Date')

# Ghép với doanh thu tháng
monthly_analysis = monthly_sales.join(monthly_inventory)

fig, ax1 = plt.subplots(figsize=(15, 6))

ax1.plot(monthly_analysis.index, monthly_analysis['Revenue'], color='royalblue', marker='o', label='Doanh thu Tháng')
ax1.set_xlabel('Thời gian')
ax1.set_ylabel('Doanh thu', color='royalblue')
ax1.tick_params(axis='y', labelcolor='royalblue')
ax1.grid(alpha=0.3)

# Cột trục 2: hiển thị số ngày Stockout
ax2 = ax1.twinx()
ax2.bar(monthly_analysis.index, monthly_analysis['stockout_days'], color='crimson', alpha=0.3, width=20, label='Tổng ngày Hết hàng (Stockouts)')
ax2.set_ylabel('Số ngày Hết hàng', color='crimson')
ax2.tick_params(axis='y', labelcolor='crimson')

fig.suptitle('Mối Tương quan Khắc nghiệt: Có Phải Đứt gãy Chuỗi Cung ứng Đã Chặn Đứng Đà Tăng Trưởng?', fontsize=14)
fig.tight_layout()
plt.show()

### 4. Tín hiệu Dẫn báo (Leading Indicator) - Web Traffic có thể báo trước Doanh thu?
Sức mạnh của phân tích chuỗi thời gian không chỉ là nhìn mốc quá khứ, mà là dùng các biến dẫn dắt để dự báo tương lai.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Chuẩn bị dữ liệu Web Traffic
web_traffic_df = web_traffic.copy()
web_traffic_df['date'] = pd.to_datetime(web_traffic_df['date'])
web_traffic_df = web_traffic_df.set_index('date')

# 2. Lọc dữ liệu năm 2022 cho cả Sales và Web Traffic
sales_2022 = sales[sales.index.year == 2022]
web_2022 = web_traffic_df[web_traffic_df.index.year == 2022]

# 3. Kết hợp dữ liệu và loại bỏ giá trị rỗng (NaN)
merged_2022 = sales_2022[['Revenue']].join(web_2022[['sessions']]).dropna()

# 4. Tính toán tương quan trễ (Cross-correlation) từ -7 đến +7 ngày
lags = range(-7, 8)
corrs = [merged_2022['sessions'].shift(lag).corr(merged_2022['Revenue']) for lag in lags]

# 5. Trực quan hóa kết quả
plt.figure(figsize=(10, 5))
plt.bar(lags, corrs, color='purple', alpha=0.6)

plt.title('Tương quan Trễ giữa Lưu Lượng Truy Cập và Doanh Thu')
plt.xlabel('Độ trễ (Days Lag)')
plt.ylabel('Hệ số tương quan (Correlation)')
plt.grid(alpha=0.2)
plt.axvline(x=0, color='red', linestyle='--', alpha=0.5) # Đường mốc lag 0
plt.tight_layout()
plt.show()

#### Key Findings (Tham khảo):
* Lưu lượng truy cập cao không nhất thiết tạo ra doanh thu ngay lập tức.
* Vùng Lag âm thể hiện tín hiệu cảnh báo sớm của Traffic lên Doanh thu vài ngày sau đó.

### 5. Chiến lược Hành động: Tối ưu hóa Kho và Khuyến mãi Từng dòng Sản phẩm
Xây dựng Ma trận ra Quyết định: Sản phẩm nào nên chạy Promo dọn kho, sản phẩm nào nên giữ giá để duy trì biên lợi nhuận?

In [ ]:
# Lấy ngày cập nhật kho gần nhất
latest_date = inventory['snapshot_date'].max()
inventory_latest = inventory[inventory['snapshot_date'] == latest_date].copy()

# Tính toán biên lợi nhuận gộp cho từng sản phẩm
products_df = products.copy()
products_df['gross_margin_pct'] = (products_df['price'] - products_df['cogs']) / products_df['price'] * 100

# Kết hợp dữ liệu kho và thông tin sản phẩm
inv_prod = pd.merge(inventory_latest, products_df[['product_id', 'gross_margin_pct', 'price']], on='product_id', how='inner')

# Trực quan hóa Ma trận Rủi ro Hàng tồn kho (Inventory Risk Matrix)
plt.figure(figsize=(14, 8))
sns.scatterplot(
    data=inv_prod, 
    x='days_of_supply', 
    y='sell_through_rate', 
    size='stock_on_hand', 
    sizes=(20, 500), 
    hue='overstock_flag', 
    palette='Set1', 
    alpha=0.7
)

# Thêm các đường ranh giới quyết định (Thresholds)
plt.axvline(x=90, color='grey', linestyle='--')  # Mốc 90 ngày tồn kho
plt.axhline(y=0.2, color='grey', linestyle='--') # Mốc tỷ lệ bán hết (Sell-through rate) 20%

# Cấu hình hiển thị
plt.xscale('log') # Sử dụng thang Log cho trục X để dễ quan sát các khoảng chênh lệch lớn
plt.title('Inventory Risk Matrix')
plt.tight_layout()
plt.show()

#### (Tham khảo) Đề xuất Hành động:

* Ngày tồn cao + Bán chậm: Cần chiến lược xả kho (Markdown) cấp tốc.

* Bán nhanh + Tồn thấp: Tiềm ẩn rủi ro Stockout (hết hàng), cần tối ưu hệ thống Re-order (tái đặt hàng) định kỳ.